In [ ]:
import numpy as np
import pandas as pd
from sklearn import linear_model
from sklearn.preprocessing import PolynomialFeatures
from tqdm.notebook import tqdm
from sklearn import ensemble
import matplotlib.pyplot as plt
import os
import pickle
import sys

In [ ]:
df = pd.read_csv('resource_consumption_dataset.csv', delimiter=';')
df_wns = pd.read_csv('WNS_dataset.csv', delimiter=';')

In [ ]:
df = df.dropna()
df = df.sample(frac=1).reset_index(drop=True)
df_wns = df_wns.dropna()
df_wns = df_wns.sample(frac=1).reset_index(drop=True)

In [ ]:
df

In [ ]:
np.random.seed(42)
remove_outliers = True
if remove_outliers:
    df = df[df['Outlier']!='Si']
    df_wns = df_wns[df_wns['Outlier']!='Si']

df = df.drop(columns=['Outlier'])
dataset = df.to_numpy()

df_wns = df_wns.drop(columns=['Outlier'])
dataset_wns = df_wns.to_numpy()

In [ ]:
normalize = False
mean = np.mean(dataset,axis=0)
std = np.std(dataset,axis=0)
mean_wns = np.mean(dataset_wns,axis=0)
std_wns = np.std(dataset_wns,axis=0)
if normalize:
    dataset = (dataset - mean)/std
    dataset_wns = (dataset_wns - mean_wns)/std_wns


In [ ]:
train_percentage = 0.8
val_percentage = 0.2

X = dataset[:,:3]
Y = dataset[:,3:]

X_wns = dataset_wns[:,:3]
Y_wns = dataset_wns[:,3:]

In [ ]:
feature_to_predict = 0 #alternatives: 0 for LUTs, 1 for FFs, 2 for BRAMs, 3 for PS8 LUTs, 4 for PS8 FFs

In [ ]:
poly_features = PolynomialFeatures(degree=3)
X_poly = poly_features.fit_transform(X)

X_train = X_poly[:int(train_percentage*X_poly.shape[0])]
X_val = X_poly[int(train_percentage*X_poly.shape[0]):int((train_percentage+val_percentage)*X_poly.shape[0])]
X_test = X_poly[int((train_percentage+val_percentage)*X_poly.shape[0]):]

Y_train = Y[:int(train_percentage*X_poly.shape[0]),feature_to_predict]
Y_val = Y[int(train_percentage*X_poly.shape[0]):int((train_percentage+val_percentage)*X_poly.shape[0]),feature_to_predict]
Y_test = Y[int((train_percentage+val_percentage)*X_poly.shape[0]):,feature_to_predict]

X_poly_wns = poly_features.fit_transform(X_wns)

indexes = np.random.choice(Y_wns.shape[0],Y_wns.shape[0],replace=True)

X_train_wns = X_poly_wns[indexes[:int(train_percentage*X_poly_wns.shape[0])]]
X_val_wns = X_poly_wns[indexes[int(train_percentage*X_poly_wns.shape[0]):int((train_percentage+val_percentage)*X_poly_wns.shape[0])]]
X_test_wns = X_poly_wns[indexes[int((train_percentage+val_percentage)*X_poly_wns.shape[0]):]]

Y_train_wns = Y_wns[indexes[:int(train_percentage*X_poly_wns.shape[0])]]
Y_val_wns = Y_wns[indexes[int(train_percentage*X_poly_wns.shape[0]):int((train_percentage+val_percentage)*X_poly_wns.shape[0])]]
Y_test_wns = Y_wns[indexes[int((train_percentage+val_percentage)*X_poly_wns.shape[0]):]]

In [ ]:
X_train.shape, X_val.shape, X_test.shape, Y_train.shape, Y_val.shape, Y_test.shape

In [ ]:
#X_train = X_train[:,[0,1,2,3,5]]
#X_val = X_val[:,[0,1,2,3,5]]
#X_test = X_test[:,[0,1,2,3,5]]

In [ ]:
linear = True
if linear:
    model = linear_model.Ridge(alpha=0.0)
    model.fit(X_train,Y_train)
else:
    model = ensemble.GradientBoostingRegressor(n_estimators=1000)
    model.fit(X_train,Y_train)

In [ ]:
predictions_val = model.predict(X_val)*std[feature_to_predict+3]+mean[feature_to_predict+3]
if not normalize:
    predictions_val = model.predict(X_val)

In [ ]:
errors = np.abs(predictions_val-(Y_val*std[feature_to_predict+3]+mean[feature_to_predict+3]))
if not normalize:
    errors = np.abs(predictions_val-Y_val)
    
errors, np.mean(errors), max(errors)

In [ ]:
X[0], X_poly[0]

In [ ]:
model.intercept_, model.coef_

# Tuning of degree and regularization coefficient for Ridge, Lasso and Elastic Net

In [ ]:
degrees = np.arange(7)
coefficients = np.arange(1000)/10000
feature_to_predict = 1
best_loss = np.inf
best_alpha = 0
best_degree = 0
best_train_metrics = []
best_metrics = []

for coeff in tqdm(coefficients):
    train_metrics = []
    metrics = []
    for deg in degrees:
        poly_features = PolynomialFeatures(degree=deg)
        X_poly = poly_features.fit_transform(X)

        X_train = X_poly[:int(train_percentage*X_poly.shape[0])]
        X_val = X_poly[int(train_percentage*X_poly.shape[0]):int((train_percentage+val_percentage)*X_poly.shape[0])]
        X_test = X_poly[int((train_percentage+val_percentage)*X_poly.shape[0]):]

        Y_train = Y[:int(train_percentage*X_poly.shape[0]),feature_to_predict]
        Y_val = Y[int(train_percentage*X_poly.shape[0]):int((train_percentage+val_percentage)*X_poly.shape[0]),feature_to_predict]
        Y_test = Y[int((train_percentage+val_percentage)*X_poly.shape[0]):,feature_to_predict]

        model = linear_model.Ridge(alpha=coeff)
        model.fit(X_train,Y_train)
        predictions_val = model.predict(X_val)*std[feature_to_predict+3]+mean[feature_to_predict+3]
        predictions_train = model.predict(X_train)*std[feature_to_predict+3]+mean[feature_to_predict+3]
        train_errors = np.abs(predictions_train-(Y_train*std[feature_to_predict+3]+mean[feature_to_predict+3]))
        errors = np.abs(predictions_val-(Y_val*std[feature_to_predict+3]+mean[feature_to_predict+3]))
        train_metrics.append(np.mean(train_errors))
        val_loss = np.mean(errors)
        metrics.append(val_loss)
        if val_loss<best_loss:
            best_loss = val_loss
            best_alpha = coeff
            best_degree = deg
            best_train_metrics = train_metrics
            best_metrics = metrics
    

In [ ]:
best_loss, best_alpha, best_degree

In [ ]:
plt.plot(best_train_metrics[:10])

In [ ]:
plt.plot(best_metrics[:10])

In [ ]:
degrees = np.arange(10)
coefficients = np.arange(1000)/1000
feature_to_predict = 0
best_loss = np.inf
best_alpha = 0
best_degree = 0
best_train_metrics = []
best_metrics = []

for coeff in tqdm(coefficients):
    train_metrics = []
    metrics = []
    for deg in degrees:
        poly_features = PolynomialFeatures(degree=deg)
        X_poly = poly_features.fit_transform(X)

        X_train = X_poly[:int(train_percentage*X_poly.shape[0])]
        X_val = X_poly[int(train_percentage*X_poly.shape[0]):int((train_percentage+val_percentage)*X_poly.shape[0])]
        X_test = X_poly[int((train_percentage+val_percentage)*X_poly.shape[0]):]

        Y_train = Y[:int(train_percentage*X_poly.shape[0]),feature_to_predict]
        Y_val = Y[int(train_percentage*X_poly.shape[0]):int((train_percentage+val_percentage)*X_poly.shape[0]),feature_to_predict]
        Y_test = Y[int((train_percentage+val_percentage)*X_poly.shape[0]):,feature_to_predict]

        model = linear_model.Lasso(alpha=coeff)
        model.fit(X_train,Y_train)
        predictions_val = model.predict(X_val)*std[feature_to_predict+3]+mean[feature_to_predict+3]
        predictions_train = model.predict(X_train)*std[feature_to_predict+3]+mean[feature_to_predict+3]
        train_errors = np.abs(predictions_train-(Y_train*std[feature_to_predict+3]+mean[feature_to_predict+3]))
        errors = np.abs(predictions_val-(Y_val*std[feature_to_predict+3]+mean[feature_to_predict+3]))
        train_metrics.append(np.mean(train_errors))
        val_loss = np.mean(errors)
        metrics.append(val_loss)
        if val_loss<best_loss:
            best_loss = val_loss
            best_alpha = coeff
            best_degree = deg
            best_train_metrics = train_metrics
            best_metrics = metrics
    

In [ ]:
best_loss, best_alpha, best_degree

In [ ]:
plt.plot(best_train_metrics[:10])

In [ ]:
plt.plot(best_metrics[:10])

In [ ]:
degrees = np.arange(7)
coefficients = np.arange(100)/1000
l1_coefficients = np.arange(100)/1000
feature_to_predict = 0
best_loss = np.inf
best_alpha = 0
best_l1 = 0
best_degree = 0
best_train_metrics = []
best_metrics = []

for coeff in tqdm(coefficients):
    for l1_coeff in (l1_coefficients):
        train_metrics = []
        metrics = []
        for deg in degrees:
            poly_features = PolynomialFeatures(degree=deg)
            X_poly = poly_features.fit_transform(X)

            X_train = X_poly[:int(train_percentage*X_poly.shape[0])]
            X_val = X_poly[int(train_percentage*X_poly.shape[0]):int((train_percentage+val_percentage)*X_poly.shape[0])]
            X_test = X_poly[int((train_percentage+val_percentage)*X_poly.shape[0]):]

            Y_train = Y[:int(train_percentage*X_poly.shape[0]),feature_to_predict]
            Y_val = Y[int(train_percentage*X_poly.shape[0]):int((train_percentage+val_percentage)*X_poly.shape[0]),feature_to_predict]
            Y_test = Y[int((train_percentage+val_percentage)*X_poly.shape[0]):,feature_to_predict]

            model = linear_model.ElasticNet(alpha=coeff, l1_ratio=l1_coeff)
            model.fit(X_train,Y_train)
            predictions_val = model.predict(X_val)*std[feature_to_predict+3]+mean[feature_to_predict+3]
            predictions_train = model.predict(X_train)*std[feature_to_predict+3]+mean[feature_to_predict+3]
            train_errors = np.abs(predictions_train-(Y_train*std[feature_to_predict+3]+mean[feature_to_predict+3]))
            errors = np.abs(predictions_val-(Y_val*std[feature_to_predict+3]+mean[feature_to_predict+3]))
            train_metrics.append(np.mean(train_errors))
            val_loss = np.mean(errors)
            metrics.append(val_loss)
            if val_loss<best_loss:
                best_loss = val_loss
                best_alpha = coeff
                best_l1 = l1_coeff
                best_degree = deg
                best_train_metrics = train_metrics
                best_metrics = metrics

In [ ]:
best_loss, best_alpha, best_l1, best_degree

In [ ]:
plt.plot(best_train_metrics[:10])

In [ ]:
plt.plot(best_metrics[:10])

## Visualization

In [ ]:
df

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for row in df.iterrows():
    row = row[1]
    x = row['total_pes']*row['n_paths']
    y = row['sample_dim']
    z = row['LUTs']
    ax.scatter(x,y,z,marker='o')

## CROSS-VALIDATION by hands

In [ ]:
def Xvalidation(X_poly, Y, std = 1, mean = 0, normalize=False,coefficients = [1e-5,1e-4,1e-3,1e-2,1e-1,0]): 
    best_coeff = 0
    best_metric = np.inf
    best_errors = []
    all_errors = []
    for coeff in coefficients:
        errors = []
        for i in range(X_poly.shape[0]):
            X_train = np.delete(X_poly,i,axis=0)
            X_val = X_poly[i:i+1]
            Y_train = np.delete(Y,i)
            Y_val = Y[i:i+1]
            model = linear_model.Ridge(alpha=coeff)
            model.fit(X_train,Y_train)
            predictions_val = model.predict(X_val)*std+mean
            if not normalize:
                predictions_val = model.predict(X_val)

            error = (predictions_val-(Y_val*std+mean))[0]
            if not normalize:
                error = (Y_val - predictions_val)[0]

            errors.append(error)
        
        metric = np.mean(np.abs(errors))
        all_errors.append(metric)
        if metric < best_metric:
            best_metric = metric
            best_coeff = coeff
            best_errors = errors

    return best_metric, best_coeff, best_errors, all_errors

## Feature selection

In [ ]:
poly_features = PolynomialFeatures(degree=3)
X_poly = poly_features.fit_transform(X)

In [ ]:
stop = False
X_initial = X_poly.copy()
feature_to_predict = 0
while(not stop):
    stop = True
    best_metric,best_coeff,best_errors,all_errors = Xvalidation(X_initial, Y[:,feature_to_predict],std=std[3+feature_to_predict],mean=mean[3+feature_to_predict],normalize=normalize)
    print(best_metric)
    for i in tqdm(range(X_initial.shape[1])):
        X_red = np.delete(X_initial, i, axis=1)
        metric,_,_,_ = Xvalidation(X_red, Y[:,feature_to_predict],std=std[3+feature_to_predict],mean=mean[3+feature_to_predict],normalize=normalize)
        if metric < best_metric:
            best_metric = metric
            stop = False
            to_remove = i
        
    if not stop:
        X_initial = np.delete(X_initial, to_remove, axis=1)
        print("feature " + str(to_remove) + " removed")
    

In [ ]:
best_metric, best_coeff

In [ ]:
poly_features = PolynomialFeatures(degree=3)
X_poly = poly_features.fit_transform(X)
feature_to_predict = 0
_,_,errors1,_ = Xvalidation(np.delete(X_poly,[0,4,10,11,12,15,16,17,18],axis=1),Y[:,feature_to_predict])
feature_to_predict = 1
_,_,errors2,_ = Xvalidation(np.delete(X_poly,[0,4,7,9,10,11,12,13,15,16,17,18],axis=1),Y[:,feature_to_predict])
feature_to_predict = 3
_,_,errors4,_ = Xvalidation(np.delete(X_poly,[3,4,6,8,9,12,13,14,15,17,18,19],axis=1),Y[:,feature_to_predict])
feature_to_predict = 4
_,_,errors5,_ = Xvalidation(np.delete(X_poly,[0,3,6,7,8,9,12,14,17,18],axis=1),Y[:,feature_to_predict])

poly_features = PolynomialFeatures(degree=2)
X_poly = poly_features.fit_transform(X)
feature_to_predict = 2
_,_,errors3,_ = Xvalidation(np.delete(X_poly,[0,3,5],axis=1),Y[:,feature_to_predict])

In [ ]:
np.max(np.abs(errors1+errors4)), np.max(np.abs(errors2+errors5)), np.max(np.abs(errors3))

In [ ]:
np.mean(np.abs(errors1+errors4)), np.mean(np.abs(errors2+errors5)), np.mean(np.abs(errors3))

In [ ]:
np.percentile(errors1+errors4, 90), np.percentile(errors2+errors5, 90), np.percentile(errors3, 90)

## Best LUTs model

In [ ]:
feature_to_predict = 0
poly_features = PolynomialFeatures(degree=3)
X_poly = poly_features.fit_transform(X)
X_final_training = np.delete(X_poly,[0,4,10,11,12,15,16,17,18], axis=1) #remove 0,4,10,11,12,15,16,17,18 for feature selection
model = linear_model.Ridge(alpha=0.0)
model.fit(X_final_training,Y[:,feature_to_predict])
with open('../automation/resource_estimation_models/LUTs_model.pkl','wb') as f:
    pickle.dump(model,f)

In [ ]:
model.coef_

## BEST FFs model

In [ ]:
feature_to_predict = 1
poly_features = PolynomialFeatures(degree=3)
X_poly = poly_features.fit_transform(X)
X_final_training = np.delete(X_poly,[0,4,7,9,10,11,12,13,15,16,17,18], axis=1) #remove [0,4,7,9,10,11,12,13,15,16,17,18] for feature selection
model = linear_model.Ridge(alpha=0.0)
model.fit(X_final_training,Y[:,feature_to_predict])
with open('../automation/resource_estimation_models/FFs_model.pkl','wb') as f:
    pickle.dump(model,f)

## BEST BRAMs model

In [ ]:
feature_to_predict = 2
poly_features = PolynomialFeatures(degree=2)
X_poly = poly_features.fit_transform(X)
X_final_training = np.delete(X_poly,[0,3,5], axis=1) #remove [0,3,5] for feature selection
model = linear_model.Ridge(alpha=0.0001)
model.fit(X_final_training,Y[:,feature_to_predict])
with open('../automation/resource_estimation_models/BRAMs_model.pkl','wb') as f:
    pickle.dump(model,f)

## BEST PS8 LUTs model

In [ ]:
feature_to_predict = 3
poly_features = PolynomialFeatures(degree=3)
X_poly = poly_features.fit_transform(X)
X_final_training = np.delete(X_poly,[3,4,6,8,9,12,13,14,15,17,18,19], axis=1) #remove [3,4,6,8,9,12,13,14,15,17,18,19] for feature selection
model = linear_model.Ridge(alpha=0.01)
model.fit(X_final_training,Y[:,feature_to_predict])
with open('../automation/resource_estimation_models/PS8LUTs_model.pkl','wb') as f:
    pickle.dump(model,f)

## BEST PS8 FFs model

In [ ]:
feature_to_predict = 4
poly_features = PolynomialFeatures(degree=3)
X_poly = poly_features.fit_transform(X)
X_final_training = np.delete(X_poly,[0,3,6,7,8,9,12,14,17,18], axis=1) #remove [0,3,6,7,8,9,12,14,17,18] for feature selection
model = linear_model.Ridge(alpha=0.0)
model.fit(X_final_training,Y[:,feature_to_predict])
with open('../automation/resource_estimation_models/PS8FFs_model.pkl','wb') as f:
    pickle.dump(model,f)

## WNS model

In [ ]:
X_train_wns.shape, X_val_wns.shape, Y_train_wns.shape, Y_val_wns.shape

In [ ]:
model = linear_model.Ridge(alpha=0.0)
model.fit(X_train_wns,Y_train_wns)

In [ ]:
predictions_val = model.predict(X_val_wns)*std_wns[3]+mean_wns[3]
if not normalize:
    predictions_val = model.predict(X_val_wns)

errors = predictions_val-(Y_val_wns*std_wns[3]+mean_wns[3])
if not normalize:
    errors = predictions_val-Y_val_wns
    
errors, np.mean(np.abs(errors)), max(errors)

In [ ]:
for i in range(Y_val_wns.shape[0]):
    print(Y_val_wns[i], predictions_val[i])

## Feature selection for WNS model

In [ ]:
poly_features = PolynomialFeatures(degree=2)
X_poly_wns = poly_features.fit_transform(X_wns)

In [ ]:
stop = False
X_initial = X_poly_wns.copy()
coefficients = [1e-3,5e-3,1e-2,3e-2,5e-2,7e-2,9e-2,1e-1,2e-1,3e-1,5e-1,7e-1,1,0]
while(not stop):
    stop = True
    best_metric,best_coeff,best_errors,all_errors = Xvalidation(X_initial, Y_wns[:,0],std=std_wns[3],mean=mean_wns[3],normalize=normalize,coefficients=coefficients)
    print(best_metric)
    for i in tqdm(range(X_initial.shape[1])):
        X_red = np.delete(X_initial, i, axis=1)
        metric,_,_,_ = Xvalidation(X_red, Y_wns[:,0],std=std_wns[3],mean=mean_wns[3],normalize=normalize)
        if metric < best_metric:
            best_metric = metric
            stop = False
            to_remove = i
        
    if not stop:
        X_initial = np.delete(X_initial, to_remove, axis=1)
        print("feature " + str(to_remove) + " removed")

In [ ]:
best_coeff

## BEST WNS model

In [ ]:
Y_wns.shape

In [ ]:
_,_,errors1,_ = Xvalidation(np.delete(X_poly_wns,[4,7,8],axis=1),Y_wns[:,0])

In [ ]:
np.mean(np.abs(errors1)), np.percentile(errors1, 90)

In [ ]:
poly_features = PolynomialFeatures(degree=2)
X_poly_wns = poly_features.fit_transform(X_wns)
X_final_training = np.delete(X_poly_wns,[4,7,8], axis=1) #remove [4,7,8] for feature selection
model = linear_model.Ridge(alpha=1.0)
model.fit(X_final_training,Y_wns[:,0])
with open('../automation/resource_estimation_models/WNS_model.pkl','wb') as f:
    pickle.dump(model,f)